In [1]:
import math
import re
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

dataset yang digunakan disini ada 4 dokumen

In [2]:
documents = {
    "D1": "Sistem komputer mengolah data untuk menghasilkan informasi.",
    "D2": "Jaringan komputer menghubungkan perangkat untuk berbagi data.",
    "D3": "Kecerdasan buatan mengolah data dan mengenali pola informasi.",
    "D4": "Sistem temu kembali mencari informasi dari kumpulan dokumen."
}

for nama, teks in documents.items():
    print(f"{nama}: {teks}")


D1: Sistem komputer mengolah data untuk menghasilkan informasi.
D2: Jaringan komputer menghubungkan perangkat untuk berbagi data.
D3: Kecerdasan buatan mengolah data dan mengenali pola informasi.
D4: Sistem temu kembali mencari informasi dari kumpulan dokumen.


preprocessing sederhana

In [3]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

def simple_preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    text_no_stopwords = stopword_remover.remove(" ".join(tokens))
    return text_no_stopwords.split()

processed = {}

for nama, teks in documents.items():
    processed[nama] = simple_preprocess(teks)
    print(nama, ":", processed[nama])

D1 : ['sistem', 'komputer', 'mengolah', 'data', 'menghasilkan', 'informasi']
D2 : ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'berbagi', 'data']
D3 : ['kecerdasan', 'buatan', 'mengolah', 'data', 'mengenali', 'pola', 'informasi']
D4 : ['sistem', 'temu', 'mencari', 'informasi', 'kumpulan', 'dokumen']


membangun bag-of-words

In [4]:
vocabulary = sorted(set(term for tokens in processed.values() for term in tokens))

bow = pd.DataFrame(
    {
        doc: [processed[doc].count(term) for term in vocabulary]
        for doc in processed
    },
    index=vocabulary
)

print("Vocabulary:")
print(vocabulary)
print("\nMatriks Bag-of-Words:")
display(bow)


Vocabulary:
['berbagi', 'buatan', 'data', 'dokumen', 'informasi', 'jaringan', 'kecerdasan', 'komputer', 'kumpulan', 'mencari', 'mengenali', 'menghasilkan', 'menghubungkan', 'mengolah', 'perangkat', 'pola', 'sistem', 'temu']

Matriks Bag-of-Words:


,D1,D2,D3,D4
berbagi,0,1,0,0
buatan,0,0,1,0
data,1,1,1,0
dokumen,0,0,0,1
informasi,1,0,1,1
jaringan,0,1,0,0
kecerdasan,0,0,1,0
komputer,1,1,0,0
kumpulan,0,0,0,1
mencari,0,0,0,1


Term Frequency (TF)

In [6]:
tf = pd.DataFrame(index=vocabulary, columns=processed.keys(), dtype=float)

for doc, tokens in processed.items():
    counts = Counter(tokens)
    total_terms = len(tokens)

    for term in vocabulary:
        tf.loc[term, doc] = counts[term] / total_terms if total_terms else 0

print("Matriks TF:")
display(tf.round(4))


Matriks TF:


,D1,D2,D3,D4
berbagi,0.0000,0.1667,0.0000,0.0000
buatan,0.0000,0.0000,0.1429,0.0000
data,0.1667,0.1667,0.1429,0.0000
dokumen,0.0000,0.0000,0.0000,0.1667
informasi,0.1667,0.0000,0.1429,0.1667
jaringan,0.0000,0.1667,0.0000,0.0000
kecerdasan,0.0000,0.0000,0.1429,0.0000
komputer,0.1667,0.1667,0.0000,0.0000
kumpulan,0.0000,0.0000,0.0000,0.1667
mencari,0.0000,0.0000,0.0000,0.1667


Document Frequency (DF) dan IDF

In [7]:
N = len(processed)

df_values = {}
idf_values = {}

for term in vocabulary:
    df = sum(1 for tokens in processed.values() if term in tokens)
    df_values[term] = df
    idf_values[term] = math.log10(N / df) if df > 0 else 0

df_idf = pd.DataFrame({
    "DF": pd.Series(df_values),
    "IDF": pd.Series(idf_values)
})

display(df_idf.round(4))


,DF,IDF
berbagi,1,0.6021
buatan,1,0.6021
data,3,0.1249
dokumen,1,0.6021
informasi,3,0.1249
jaringan,1,0.6021
kecerdasan,1,0.6021
komputer,2,0.3010
kumpulan,1,0.6021
mencari,1,0.6021


TF-IDF manual

In [8]:
tfidf_manual = tf.copy()

for term in vocabulary:
    tfidf_manual.loc[term] = tf.loc[term] * idf_values[term]

print("Matriks TF-IDF Manual:")
display(tfidf_manual.round(4))


Matriks TF-IDF Manual:


,D1,D2,D3,D4
berbagi,0.0000,0.1003,0.0000,0.0000
buatan,0.0000,0.0000,0.0860,0.0000
data,0.0208,0.0208,0.0178,0.0000
dokumen,0.0000,0.0000,0.0000,0.1003
informasi,0.0208,0.0000,0.0178,0.0208
jaringan,0.0000,0.1003,0.0000,0.0000
kecerdasan,0.0000,0.0000,0.0860,0.0000
komputer,0.0502,0.0502,0.0000,0.0000
kumpulan,0.0000,0.0000,0.0000,0.1003
mencari,0.0000,0.0000,0.0000,0.1003


Implementasi TF-IDF dengan Scikit-Learn

In [9]:
processed_texts = [" ".join(processed[doc]) for doc in documents]

vectorizer = TfidfVectorizer(
    lowercase=False,
    norm=None,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False
)

sk_matrix = vectorizer.fit_transform(processed_texts)

tfidf_sklearn = pd.DataFrame(
    sk_matrix.toarray().T,
    index=vectorizer.get_feature_names_out(),
    columns=documents.keys()
)

print("Matriks TF-IDF dari scikit-learn:")
display(tfidf_sklearn.round(4))


Matriks TF-IDF dari scikit-learn:


,D1,D2,D3,D4
berbagi,0.0000,2.3863,0.0000,0.0000
buatan,0.0000,0.0000,2.3863,0.0000
data,1.2877,1.2877,1.2877,0.0000
dokumen,0.0000,0.0000,0.0000,2.3863
informasi,1.2877,0.0000,1.2877,1.2877
jaringan,0.0000,2.3863,0.0000,0.0000
kecerdasan,0.0000,0.0000,2.3863,0.0000
komputer,1.6931,1.6931,0.0000,0.0000
kumpulan,0.0000,0.0000,0.0000,2.3863
mencari,0.0000,0.0000,0.0000,2.3863


membandingkan manual dan Scikit-Learn

In [10]:
tfidf_sklearn = tfidf_sklearn.reindex(vocabulary)

comparison = tfidf_manual - tfidf_sklearn

print("Selisih TF-IDF Manual - Scikit-Learn:")
display(comparison.round(6))

print("Nilai maksimum selisih absolut:",
      comparison.abs().to_numpy().max())


Selisih TF-IDF Manual - Scikit-Learn:


,D1,D2,D3,D4
berbagi,0.000000,-2.285951,0.000000,0.000000
buatan,0.000000,0.000000,-2.300286,0.000000
data,-1.266859,-1.266859,-1.269834,0.000000
dokumen,0.000000,0.000000,0.000000,-2.285951
informasi,-1.266859,0.000000,-1.269834,-1.266859
jaringan,0.000000,-2.285951,0.000000,0.000000
kecerdasan,0.000000,0.000000,-2.300286,0.000000
komputer,-1.642976,-1.642976,0.000000,0.000000
kumpulan,0.000000,0.000000,0.000000,-2.285951
mencari,0.000000,0.000000,0.000000,-2.285951


Nilai maksimum selisih absolut: 2.300285790930182


Term dengan bobot TF-IDF tertinggi

In [11]:
for doc in documents.keys():
    max_term = tfidf_manual[doc].idxmax()
    max_value = tfidf_manual[doc].max()
    print(f"{doc}: term tertinggi = '{max_term}', bobot TF-IDF = {max_value:.4f}")

D1: term tertinggi = 'menghasilkan', bobot TF-IDF = 0.1003
D2: term tertinggi = 'berbagi', bobot TF-IDF = 0.1003
D3: term tertinggi = 'buatan', bobot TF-IDF = 0.0860
D4: term tertinggi = 'dokumen', bobot TF-IDF = 0.1003


Analisis

Bobot Tf-IDF memberikan nilai yang lebih besar kepada term yang muncul dalam suatu dokumen tetapi tidak muncul pada banyak dokumen lain, maka dengan itu term dengan bobot tertinggi dapat menjadi representasi yang lebih khas untuk dokumen tersebut. pada dataset yang di pakai disini dengan istilah sistem, jaringan, kecerdasan, atau informasi bisa memiliki bobot yang berbeda bergantung pada frekuensi kemunculan dan jumlah dokumen yang mengandung istilah tersebut. hasil manual dan scikit-learn dapat dibandingkan melalui matriks selisih dengan konfigurasi scikit-learn yang disesuaikan, hasilnya mengikuti formula manual yang digunakan.